# ¿La selección por pares también gana en **aciertos**? `obs+recon` vs `energy+recon` — LFM3-1.7B, GSM8K (Colab L4)

En Qwen3.5-0.8B, elegir las neuronas **por pares** y luego compensar (`obs+recon`) le ganó a `energy+recon` en
perplejidad de mate en todos los presupuestos (30 %: 6.97 vs 7.17). Pero perplejidad ≠ aciertos. Aquí se mide lo que
importa: **aciertos en GSM8K** con el modelo recortado de verdad (matrices físicamente más chicas), mismo setup que
la tabla del README (LFM3-1.7B, greedy, 512 tokens, Gram de 20k tokens de mate). Antes: `energy+recon` 79.0 / 65.5 /
37.0 % a 70 / 50 / 30 % (modelo completo 83.0 %, 200 problemas).

Ahora **400 problemas** y los dos índices en la misma corrida, con resultado por problema → comparación **pareada**.
`obs` = elimina neuronas una por una, la que menos daño hace después de reajustar a las demás (usa quién se activa
junto con quién).

~45–60 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/task_index_obs_lfm3.json`. Si se corta, retoma (Drive).
Al final el runtime se apaga solo.

In [ ]:
N_PROBLEMS = 400
BATCH      = 32
MAX_NEW    = 512
BUDGETS    = "0.7,0.5,0.3"
INDEXES    = "math-energy+recon,math-obs+recon"
GRAM_TOK   = 20000
RUN_NAME   = "task_index_obs_lfm3"
REPO       = "/content/wsi"
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(FINALES, exist_ok=True)
OUT = f"{DRIVE}/results.json"   # se escribe tras cada config: si se corta, retoma

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers huggingface_hub
!python -c "from transformers.integrations import use_kernel_func_from_hub_with_fallback; from transformers.integrations.accelerate import force_accelerate_hooks" 2>/dev/null && echo "transformers OK" || pip install -q -U git+https://github.com/huggingface/transformers
!python -c "import transformers; print('transformers', transformers.__version__)"

In [ ]:
import subprocess
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert "math-obs" in open(f"{REPO}/scripts/experiment_moi_task_index_lfm3.py").read(), "falta push del índice obs"
print("commit", COMMIT)

In [ ]:
!cd {REPO}/scripts && python -u experiment_moi_task_index_lfm3.py --model-dir LiquidAI/LFM3-1.7B --data-dir /content/data \
    --n-problems {N_PROBLEMS} --batch {BATCH} --max-new-tokens {MAX_NEW} --budgets {BUDGETS} --indexes {INDEXES} \
    --gram-tokens {GRAM_TOK} --out {OUT}

In [ ]:
import json, random, datetime
R = json.load(open(OUT))
r = R["results"]
f = r["full"]["gsm8k"]

def paired(a, b, iters=10000, seed=0):
    d = [x - y for x, y in zip(a, b)]
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(d) for _ in d) / len(d) for _ in range(iters))
    return {"diff": round(sum(d) / len(d), 4), "ci95": [round(boots[int(0.025 * iters)], 4), round(boots[int(0.975 * iters)], 4)],
            "a_only": sum(x > y for x, y in zip(a, b)), "b_only": sum(y > x for x, y in zip(a, b))}

print(f"modelo completo: {f['accuracy']:.1%} ({f['n']} problemas)\n")
print(f"{'neuronas':>8s} {'energy+recon':>13s} {'obs+recon':>10s} {'obs - energy':>13s} {'IC 95%':>17s} {'solo obs/solo energy':>21s}")
summary = {}
for b in BUDGETS.split(","):
    e, o = r[f"math-energy+recon@{float(b)}"], r[f"math-obs+recon@{float(b)}"]
    pc = paired(o["gsm8k"]["per_problem"], e["gsm8k"]["per_problem"])
    summary[b] = {"energy_recon": e["gsm8k"]["accuracy"], "obs_recon": o["gsm8k"]["accuracy"], "obs_vs_energy": pc,
                  "ppl_math": [e["ppl_gsm8k_solutions"], o["ppl_gsm8k_solutions"]],
                  "ppl_wiki": [e["ppl_wikitext"], o["ppl_wikitext"]]}
    print(f"{float(b):8.0%} {e['gsm8k']['accuracy']:13.1%} {o['gsm8k']['accuracy']:10.1%} {pc['diff']:+13.1%} "
          f"[{pc['ci95'][0]:+.1%}, {pc['ci95'][1]:+.1%}] {pc['a_only']:>10d}/{pc['b_only']}")
R["summary"] = summary
R["_final"] = {"complete": True, "commit": COMMIT, "notebook": "task_index_obs_lfm3_colab.ipynb",
               "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
               "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(R, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")